# Myanmar Weather Intelligence
## A GPU-powered NVIDIA Earth-2 demonstration

**Project overview**

Myanmar Weather Intelligence demonstrates how pretrained NVIDIA FourCastNet models can transform real atmospheric initial conditions into regional weather forecasts. Built with Earth2Studio and an interactive dashboard, the application runs inference on a Google Colab GPU and presents temperature and wind predictions for selected Myanmar cities.

**မြန်မာနိုင်ငံအတွက် မိုးလေဝသ AI သရုပ်ပြစနစ်** — Real GFS input data ကို အသုံးပြုပြီး NVIDIA FourCastNet model ကို GPU ပေါ်တွင် run ကာ အပူချိန်နှင့် လေတိုက်နှုန်း ခန့်မှန်းချက်များကို dashboard ဖြင့် ပြသသည်။

| Item | Demonstration scope |
|---|---|
| AI model | NVIDIA FourCastNet (FCN), pretrained weights |
| Inference toolkit | NVIDIA Earth2Studio |
| Initial conditions | NOAA GFS atmospheric fields |
| Execution | Google Colab CUDA GPU |
| Regional outputs | 2 m air temperature and 10 m wind speed |
| Future forecast window | Next 6/12 hours or 1–14 days from the request time, on a 6-hour model grid |
| Interface | Interactive Gradio dashboard with result exports |
| Intended use | Research and software showcase; no local accuracy claim |

### Architecture
Download model weights → retrieve real GFS initial conditions → execute FCN on GPU → select Myanmar outputs → display maps, city trends and downloadable evidence.

### Run in Google Colab
1. Upload this notebook at https://colab.research.google.com/.
2. Choose **Runtime → Change runtime type → A100 GPU** or another suitable GPU with sufficient memory. Current NVIDIA FCN catalog metadata lists a 40 GB GPU; a free T4 may not fit.
3. Run the installation cell once. If it fails, inspect `/content/earth2_fcn_install.log` before proceeding.
4. After installation succeeds, **restart the session** and continue from **Imports and GPU check**.
5. Run the backend and dashboard cells. Start with a **6-hour horizon**, then extend the forecast after a successful first run.

**Two separate forecast tabs:**
- **Run NVIDIA AI forecast:** predict 6/12/24/30/48 hours after the selected GFS initialization. **30 hours** is the default because the local showcase timeline is 0, 6, 12, 18, 24, 30. Historical cases are allowed when the archive has data. The table includes +0 h and all model steps. Download the zip and drop it on the showcase UI.
- **Predict future weather:** extend the rollout to cover 6/12 hours or **1–14 days (24–336 hours)** after the request time. The showcase UI plays every 6-hour step in the zip, including a 14-day run. Longer rollouts take more GPU time and are more uncertain. The Colab table contains future points only; the zip still keeps every step.

**Next-day selection:** choose **Next day**, **Next 2 days**, … **Next 14 days** under **Future prediction — select next day(s)**, then press **Predict future weather**. Each day means 24 hours from the request time, rather than a calendar-day summary. The default is **Next day — 24 hours**. **Next 14 days — 336 hours** is supported. The **Run NVIDIA AI forecast** tab retains its own 6–48 hour control (30 hours matches the short showcase timeline). Each tab preserves its own city, initialization, charts, tables and downloads when switching tabs.

**Future forecasting:** `AUTO` checks GFS cycles newest first and selects the latest cycle with a published 0.25° initial-condition file and index. It extends the model rollout far enough to cover the selected number of hours beyond the request time. The final valid time is rounded forward to the FCN 6-hour grid. The dashboard table shows only timestamps later than display generation; exports retain the initial state and all forecast steps for provenance.

**Product scope:** this is short-range weather inference. Long-term climate projection, rainfall prediction and StormScope satellite nowcasting are outside this notebook's scope. Maps show a bounding box containing Myanmar and neighboring areas. Outputs have not been evaluated against Myanmar observations.


## Installation — FCN dependencies only
This notebook does not install StormScope extras or request Earth2Grid/NATTEN. Use a fresh Colab runtime to avoid mixing the earlier unsuccessful StormScope environment with this demo.


In [ ]:
import sys
import subprocess
from pathlib import Path

log = Path("/content/earth2_fcn_install.log")
command = [sys.executable, "-m", "pip", "install", "--upgrade",
           "earth2studio[fcn]==0.20.0", "gradio>=5,<6", "matplotlib", "pandas>=2.2,<3"]
print("Installing FCN dependencies. Full output:", log, flush=True)
with log.open("w") as stream:
    result = subprocess.run(command, stdout=stream, stderr=subprocess.STDOUT)
print("\n".join(log.read_text(errors="replace").splitlines()[-70:]))
if result.returncode:
    raise RuntimeError("Installation failed. Stop and inspect /content/earth2_fcn_install.log.")
print("Installation succeeded. Restart session; then continue from Imports and GPU check.")


Installing FCN dependencies. Full output: /content/earth2_fcn_install.log
    Found existing installation: tomlkit 0.14.0
    Uninstalling tomlkit-0.14.0:
      Successfully uninstalled tomlkit-0.14.0
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.50.1
    Uninstalling pydantic_core-2.50.1:
      Successfully uninstalled pydantic_core-2.50.1
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: psutil
    Found existing installation: psutil 5.9.5
    Uninstalling psutil-5.9.5:
      Successfully uninstalled psutil-5.9.5
  Attempting uninstall: omegaconf
    Found existing installation: omegaconf 2.4.0
    Uninstalling omegaconf-2.4.0:
      Successfully uninstalled omegaconf-2.4.0
  Attempting uninstall: h5py
    Found existing installation: h5py 3.14.0
    Uninstalling h5py-3.14.0:
      Successfully uninstalled h5py-3

## Imports and GPU check


In [ ]:
import os
os.environ.setdefault("EARTH2STUDIO_CACHE", "/content/earth2_fcn_cache")
import gc
import json
import time
import tempfile
import zipfile
import traceback
import math
import urllib.request
import urllib.error
from zoneinfo import ZoneInfo
import matplotlib.dates as mdates
from pathlib import Path
from datetime import datetime, timedelta, timezone
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import gradio as gr
from earth2studio import run
from earth2studio.data import GFS
from earth2studio.io import ZarrBackend
from earth2studio.models.px import FCN

if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU under Runtime → Change runtime type.")
DEVICE = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
print("GPU:", props.name, f"({props.total_memory / 2**30:.1f} GiB)")
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("Earth2Studio:", version("earth2studio"))
# City forecast pingpoints — coordinates match the dashboard's Myanmar city list
CITIES = {
    "Yangon": (16.8661, 96.1951),
    "Mandalay": (21.9750, 96.0833),
    "Nay Pyi Taw": (19.7633, 96.0785),
    "Taunggyi": (20.7833, 97.0333),
    "Mawlamyine": (16.4905, 97.6283),
    "Bago": (17.3221, 96.4660),
    "Pathein": (16.7797, 94.7324),
    "Sittwe": (20.1462, 92.8983),
    "Monywa": (22.1086, 95.1378),
    "Myitkyina": (25.3833, 97.4000),
    "Dawei": (14.0833, 98.2000),
    "Pyin Oo Lwin": (22.0333, 96.4667),
}
# Keep legacy dropdown key working
CITIES["Naypyidaw"] = CITIES["Nay Pyi Taw"]

# Storm-alert detection zone from showcase backend/fourcastnet_pipeline.py
# Scans Bay of Bengal: Lat 10–22°N, Lon 92–96°E; cyclone if wind >= 65 km/h
MYANMAR_BOUNDS = {"min_lat": 9.0, "max_lat": 29.0, "min_lon": 92.0, "max_lon": 102.0}
BAY_OF_BENGAL_BOUNDS = {"min_lat": 10.0, "max_lat": 22.0, "min_lon": 92.0, "max_lon": 96.0}
CYCLONE_WIND_THRESHOLD_KMH = 65.0  # IMD / DMH cyclonic storm floor

# Coastal / open-water pingpoints used for Myanmar storm-alert monitoring
# (inside the same BoB box the git showcase scans)
STORM_ALERT_PINGPOINTS = {
    "BoB Open Water (eye watch)": (15.50, 93.50),
    "Sittwe (Rakhine)": (20.1462, 92.8983),
    "Thandwe (Rakhine)": (18.4600, 94.3500),
    "Kyaukpyu (Rakhine)": (19.4300, 93.5500),
    "Pathein (Ayeyarwady)": (16.7797, 94.7324),
    "Hainggyi / Delta mouth": (15.9800, 94.3200),
    "Yangon approaches": (16.4500, 95.8500),
    "Mottama Gulf": (16.2000, 96.0000),
}

DEFAULT_DATE = "AUTO"
_MODEL = None
_RESULTS = {}


GPU: NVIDIA A100-SXM4-40GB (39.5 GiB)
PyTorch: 2.11.0+cu130 CUDA: 13.0
Earth2Studio: 0.20.0


## GPU inference backend
FCN runs globally at its native grid. Saving regional temperature/wind does not turn it into a regional model or reduce its global inference memory requirement. The model is cached for subsequent runs; the last forecast is reused when only the selected city changes. Cached provenance retains the actual original run time.


In [ ]:
def utc_now():
    return datetime.now(timezone.utc)


def cycle_urls(start):
    prefix = ("https://noaa-gfs-bdp-pds.s3.amazonaws.com/"
              f"gfs.{start:%Y%m%d}/{start:%H}/atmos/gfs.t{start:%H}z.pgrb2.0p25.f000")
    return prefix, prefix + ".idx"


def cycle_published(start):
    # S3 publishes each object atomically. Check the initial-state GRIB and index.
    # This checks object publication; the real GFS fetch still validates variables.
    for url in cycle_urls(start):
        try:
            request = urllib.request.Request(url, method="HEAD")
            with urllib.request.urlopen(request, timeout=12) as response:
                if int(response.headers.get("Content-Length", "0")) <= 0:
                    return False
        except urllib.error.HTTPError as exc:
            if exc.code == 404:
                return False
            raise RuntimeError(f"GFS publication check failed: HTTP {exc.code}.") from exc
        except (urllib.error.URLError, TimeoutError) as exc:
            raise RuntimeError("GFS publication check failed because of a network error; no forecast was generated.") from exc
    return True


def find_latest_cycle(now=None, probe=None):
    now = now or utc_now()
    probe = probe or cycle_published
    newest = now.replace(hour=(now.hour//6)*6, minute=0, second=0, microsecond=0)
    for offset in range(6):
        candidate = newest-timedelta(hours=6*offset)
        if now-candidate > timedelta(hours=30):
            break
        print("Checking GFS cycle:", candidate.isoformat(), flush=True)
        if probe(candidate):
            return candidate
    raise RuntimeError("No published GFS initial-state cycle found within the recent 30-hour search window.")


def validate(date_text, hours, city):
    if int(hours) not in (6,12,24,48,72,96,120,144,168,192,216,240,264,288,312,336) or city not in CITIES:
        raise ValueError("Choose a listed city and future forecast window.")
    text = date_text.strip()
    now = utc_now()
    if text.upper() == "AUTO":
        start = find_latest_cycle(now)
    else:
        try:
            start = datetime.strptime(text, "%Y-%m-%d %H:%M").replace(tzinfo=timezone.utc)
        except (ValueError, AttributeError):
            raise ValueError("Use AUTO, or YYYY-MM-DD HH:MM in UTC (00:00, 06:00, 12:00 or 18:00).")
        if start.hour not in (0,6,12,18) or start.minute != 0:
            raise ValueError("GFS initialization must be 00:00, 06:00, 12:00 or 18:00 UTC.")
        if not timedelta(0) <= now-start <= timedelta(hours=30):
            raise ValueError("For future forecasting choose a published initialization within the last 30 hours, or AUTO.")
        if not cycle_published(start):
            raise ValueError("That GFS cycle is not fully published yet. Choose AUTO or an earlier cycle.")
    return start, int(hours)


def future_plan(start, ahead, now=None):
    now = now or utc_now()
    target = now+timedelta(hours=int(ahead))
    rollout_hours = int(math.ceil((target-start).total_seconds()/21600))*6
    # 14 days (336 h) from request time, plus up to ~30 h of GFS-cycle lag, stays under 384 h.
    if not 6 <= rollout_hours <= 384:
        raise ValueError("Required rollout exceeds this demo's 384-hour limit (14 days plus GFS lag). Choose a newer cycle or a shorter window.")
    return rollout_hours, target


def validate_standard(date_text, hours, city):
    if int(hours) not in (6,12,24,30,48) or city not in CITIES:
        raise ValueError("Choose a listed city and forecast duration.")
    text=date_text.strip()
    now=utc_now()
    if text.upper()=="AUTO":
        start=find_latest_cycle(now)
    else:
        try:
            start=datetime.strptime(text,"%Y-%m-%d %H:%M").replace(tzinfo=timezone.utc)
        except (ValueError,AttributeError):
            raise ValueError("Use AUTO or YYYY-MM-DD HH:MM in UTC.")
        if start.hour not in (0,6,12,18) or start.minute:
            raise ValueError("Use a GFS cycle at 00:00, 06:00, 12:00 or 18:00 UTC.")
        if not datetime(2021,1,1,tzinfo=timezone.utc)<=start<=now:
            raise ValueError("Initialization must be between 2021-01-01 and now.")
        if not cycle_published(start):
            raise ValueError("The selected GFS cycle is not published or available.")
    return start,int(hours)


def predict(start, hours):
    global _MODEL
    key = (start.isoformat(), hours)
    if key in _RESULTS:
        return _RESULTS[key]
    began = time.perf_counter()
    steps = hours // 6
    try:
        if _MODEL is None:
            print("Downloading/loading NVIDIA pretrained FCN weights...")
            _MODEL = FCN.load_model(FCN.load_default_package())
            _MODEL = _MODEL.to(DEVICE).float().eval()
        with torch.inference_mode():
            output = run.deterministic(
                [start.strftime("%Y-%m-%dT%H:%M:%S")], steps, _MODEL, GFS(), ZarrBackend(),
                output_coords={"variable": np.array(["t2m", "u10m", "v10m"])},
                device=DEVICE,
            )
        lat0=np.asarray(output["lat"][:]); lon0=np.asarray(output["lon"][:])
        iy=np.flatnonzero((lat0>=9)&(lat0<=29))
        ix=np.flatnonzero((lon0>=92)&(lon0<=102))
        if not len(iy) or not len(ix):
            raise RuntimeError("Expected Myanmar regional coordinates were not returned.")
        # Showcase map samples row 0 = 29°N down to 9°N, col 0 = 92°E across to 102°E, step 0.25°.
        iy=iy[np.argsort(-lat0[iy])]
        ix=ix[np.argsort(lon0[ix])]
        lat_show=np.round(lat0[iy], 5)
        lon_show=np.round(lon0[ix], 5)
        lat_expected=np.round(np.arange(29.0, 8.875, -0.25), 5)
        lon_expected=np.round(np.arange(92.0, 102.125, 0.25), 5)
        if lat_show.shape != lat_expected.shape or not np.allclose(lat_show, lat_expected, atol=1e-3):
            raise RuntimeError(f"Latitude axis {lat_show[0]}..{lat_show[-1]} ({len(lat_show)}) is not the showcase 29°N→9°N / 0.25° / 81-row grid.")
        if lon_show.shape != lon_expected.shape or not np.allclose(lon_show, lon_expected, atol=1e-3):
            raise RuntimeError(f"Longitude axis {lon_show[0]}..{lon_show[-1]} ({len(lon_show)}) is not the showcase 92°E→102°E / 0.25° / 41-col grid.")
        def regional(variable):
            return np.stack([np.asarray(output[variable][0,i])[np.ix_(iy,ix)]
                             for i in range(steps+1)]).astype(np.float32)
        temp=regional("t2m")-273.15
        u=regional("u10m"); v=regional("v10m")
        wind=np.hypot(u,v)*3.6  # m/s -> km/h
        if not np.isfinite(temp).all() or not np.isfinite(wind).all():
            raise RuntimeError("Model output includes non-finite values.")
        torch.cuda.synchronize()
        result=dict(lat=lat0[iy],lon=lon0[ix],temp_c=temp,wind_kmh=wind,
                    lead_hours=np.arange(steps+1)*6,
                    completed_utc=datetime.now(timezone.utc).isoformat(),
                    end_to_end_seconds=time.perf_counter()-began)
        del output
    except torch.cuda.OutOfMemoryError as exc:
        _MODEL=None
        gc.collect(); torch.cuda.empty_cache()
        raise RuntimeError("GPU memory exhausted. Restart Colab or use a larger GPU. No substitute forecast was generated.") from exc
    _RESULTS.clear(); _RESULTS[key]=result
    return result


def nearest_grid_index(lats, lons, lat, lon):
    iy = int(np.argmin(np.abs(np.asarray(lats) - lat)))
    ix = int(np.argmin(np.abs(np.asarray(lons) - lon)))
    return iy, ix


def scan_storm_alert_bay_of_bengal(r, valid_times):
    """Match showcase Step 6: scan BoB box Lat 10–22 / Lon 92–96 for wind >= 65 km/h.

    Also samples STORM_ALERT_PINGPOINTS so coastal Myanmar locations are explicit
    pingpoints in the alert payload and on the forecast maps.
    """
    lats = np.asarray(r["lat"])
    lons = np.asarray(r["lon"])
    wind = np.asarray(r["wind_kmh"])  # [step, lat, lon]
    lead_hours = list(r["lead_hours"])

    lat_mask = (lats >= BAY_OF_BENGAL_BOUNDS["min_lat"]) & (lats <= BAY_OF_BENGAL_BOUNDS["max_lat"])
    lon_mask = (lons >= BAY_OF_BENGAL_BOUNDS["min_lon"]) & (lons <= BAY_OF_BENGAL_BOUNDS["max_lon"])
    if not lat_mask.any() or not lon_mask.any():
        raise RuntimeError("Bay of Bengal storm-alert window is outside the regional grid.")

    peak_wind = 0.0
    peak_coord = {"lat": 15.0, "lon": 93.5}
    peak_step = 0
    peak_timestamp = ""
    ping_samples = []

    for s_idx in range(wind.shape[0]):
        region = wind[s_idx][np.ix_(lat_mask, lon_mask)]
        flat_i = int(np.nanargmax(region))
        local_r, local_c = np.unravel_index(flat_i, region.shape)
        region_lats = lats[lat_mask]
        region_lons = lons[lon_mask]
        val = float(region[local_r, local_c])
        if val > peak_wind:
            peak_wind = val
            peak_coord = {
                "lat": round(float(region_lats[local_r]), 2),
                "lon": round(float(region_lons[local_c]), 2),
            }
            peak_step = int(lead_hours[s_idx]) if s_idx < len(lead_hours) else s_idx * 6
            peak_timestamp = (
                valid_times[s_idx].strftime("%Y-%m-%dT%H:%M:%SZ")
                if s_idx < len(valid_times) else ""
            )

    # Explicit coastal / open-water pingpoints (same watch locations as the showcase HUD)
    for name, (plat, plon) in STORM_ALERT_PINGPOINTS.items():
        iy, ix = nearest_grid_index(lats, lons, plat, plon)
        series = wind[:, iy, ix]
        s_max = int(np.nanargmax(series))
        w_max = float(series[s_max])
        in_box = (
            BAY_OF_BENGAL_BOUNDS["min_lat"] <= plat <= BAY_OF_BENGAL_BOUNDS["max_lat"]
            and BAY_OF_BENGAL_BOUNDS["min_lon"] <= plon <= BAY_OF_BENGAL_BOUNDS["max_lon"]
        )
        ping_samples.append({
            "name": name,
            "lat": plat,
            "lon": plon,
            "in_detection_box": in_box,
            "max_wind_kmh": round(w_max, 1),
            "lead_hour": int(lead_hours[s_max]) if s_max < len(lead_hours) else s_max * 6,
            "grid_lat": float(lats[iy]),
            "grid_lon": float(lons[ix]),
        })

    is_alert_active = peak_wind >= CYCLONE_WIND_THRESHOLD_KMH
    alert = {
        "alert_active": is_alert_active,
        "severity": "CYCLONIC_STORM" if is_alert_active else "NORMAL_MONSOON",
        "region": "Bay of Bengal & Coastal Myanmar",
        "region_mm": "ဘင်္ဂလားပင်လယ်အော်နှင့် မြန်မာ့ကမ်းရိုးတန်း ရေပြင်",
        "max_sustained_wind_kmh": round(peak_wind, 1),
        "threshold_kmh": CYCLONE_WIND_THRESHOLD_KMH,
        "coordinates": peak_coord,
        "timestamp": peak_timestamp or utc_now().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "lead_hour": peak_step,
        "detection_box": BAY_OF_BENGAL_BOUNDS,
        "pingpoints": ping_samples,
        "advisory_en": (
            "Cyclone Warning: Deep depression/cyclonic wind detected in the Bay of Bengal approaching Ayeyarwady/Rakhine coast. Marine operations should suspend."
            if is_alert_active
            else "Bay of Bengal wind conditions are within normal limits (below 65 km/h threshold)."
        ),
        "advisory_mm": (
            "ဆိုင်ကလုန်းမုန်တိုင်း အရေးပေါ် သတိပေးချက် - ဘင်္ဂလားပင်လယ်အော်တွင် လေတိုက်နှုန်း ၆၅ ကီလိုမီတာ/နာရီကျော် မုန်တိုင်းငယ်ဖြစ်ပေါ်နေသဖြင့် ကမ်းနီးကမ်းဝေး ရေယာဉ်များနှင့် ကမ်းရိုးတန်းနေပြည်သူများ သတိပြုကြပါရန်။"
            if is_alert_active
            else "ဘင်္ဂလားပင်လယ်အော် လေတိုက်နှုန်းမှာ ပုံမှန်အခြေအနေတွင် ရှိပါသည် (၆၅ ကီလိုမီတာ/နာရီအောက်)။"
        ),
        "generated_at": utc_now().isoformat(),
    }
    return alert


def render_result(start, hours, city, r):
    display_now = utc_now()
    mode=r.get("forecast_mode","future")
    is_future_mode=mode=="future"
    clat,clon=CITIES[city]
    iy=int(np.argmin(np.abs(r["lat"]-clat))); ix=int(np.argmin(np.abs(r["lon"]-clon)))
    valid=pd.DatetimeIndex([start+timedelta(hours=int(h)) for h in r["lead_hours"]])
    if is_future_mode and valid[-1].to_pydatetime() <= display_now:
        raise RuntimeError("The forecast is no longer in the future. Run again using AUTO.")
    local=valid.tz_convert("Asia/Yangon")
    future_mask=np.asarray(valid > pd.Timestamp(display_now))
    table=pd.DataFrame({"lead_hours":r["lead_hours"],
        "valid_utc":valid.strftime("%Y-%m-%d %H:%M UTC"),
        "valid_myanmar":local.strftime("%Y-%m-%d %H:%M MMT"),
        "city":city,"forecast_mode":mode,"temperature_c":np.round(r["temp_c"][:,iy,ix],2),
        "wind_kmh":np.round(r["wind_kmh"][:,iy,ix],2),
        "is_future_at_display":future_mask,
        "hours_ahead_at_display":np.round((valid-pd.Timestamp(display_now)).total_seconds()/3600,2),
        "type":["GFS initial state" if h==0 else "FCN forecast" for h in r["lead_hours"]]})
    future_table=table.loc[future_mask].reset_index(drop=True)
    display_table=future_table if is_future_mode else table
    display_mask=future_mask if is_future_mode else np.ones(len(table),dtype=bool)
    title=f"NVIDIA FCN | Valid {local[-1].strftime('%d %b %Y %H:%M')} MMT"
    valid_py = [ts.to_pydatetime() for ts in valid]
    storm_alert = scan_storm_alert_bay_of_bengal(r, valid_py)

    fig,axes=plt.subplots(1,2,figsize=(13,7),facecolor="#f8fafc")
    box = BAY_OF_BENGAL_BOUNDS
    for ax,field,cmap,label in [(axes[0],r["temp_c"][-1],"RdYlBu_r","2 m temperature (°C)"),
                                (axes[1],r["wind_kmh"][-1],"viridis","10 m wind speed (km/h)")]:
        image=ax.pcolormesh(r["lon"],r["lat"],field,cmap=cmap,shading="auto")
        fig.colorbar(image,ax=ax,label=label,shrink=.8)
        # Storm-alert detection box (same as git showcase pipeline)
        ax.add_patch(plt.Rectangle(
            (box["min_lon"], box["min_lat"]),
            box["max_lon"] - box["min_lon"],
            box["max_lat"] - box["min_lat"],
            fill=False, linestyle="--", linewidth=1.4, edgecolor="#e11d48",
            label="BoB storm scan",
        ))
        # Coastal / open-water storm pingpoints
        for pname, (plat, plon) in STORM_ALERT_PINGPOINTS.items():
            ax.scatter(plon, plat, c="#e11d48", s=36, marker="x", zorder=5)
            ax.annotate(pname.split(" (")[0], (plon, plat), xytext=(3, 3),
                        textcoords="offset points", fontsize=7, color="#9f1239")
        # Selected city pin
        ax.scatter(clon,clat,c="black",s=36,zorder=6)
        ax.annotate(city,(clon,clat),xytext=(4,4),textcoords="offset points")
        # Peak storm coordinate when alert is active
        if storm_alert["alert_active"]:
            eye = storm_alert["coordinates"]
            ax.scatter(eye["lon"], eye["lat"], c="#f59e0b", s=90, marker="*", zorder=7,
                       label="Peak wind")
        ax.set(xlabel="Longitude (°E)",ylabel="Latitude (°N)",title=label)
        ax.grid(alpha=.15)
    alert_tag = (
        f" | CYCLONE ALERT {storm_alert['max_sustained_wind_kmh']:.0f} km/h"
        if storm_alert["alert_active"] else " | storm scan clear"
    )
    fig.suptitle(title + alert_tag,fontsize=13,fontweight="bold",color="#123047");fig.tight_layout()
    trend,ax=plt.subplots(figsize=(10,4))
    future_times=local[display_mask].to_pydatetime()
    ax.plot(future_times,display_table.temperature_c,"o-",color="#d97706",label="Temperature")
    ax.set(xlabel="Valid time — Myanmar (UTC+06:30)",ylabel="Temperature (°C)",title=f"{city} · {'Future' if is_future_mode else 'Initialization-based'} temperature and wind")
    ax.set_facecolor("#f8fafc")
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b\n%H:%M',tz=ZoneInfo("Asia/Yangon")))
    ax2=ax.twinx();ax2.plot(future_times,display_table.wind_kmh,"s--",color="#008b8b",label="Wind")
    ax2.set_ylabel("Wind (km/h)")
    ax.legend(loc="upper left",frameon=False);ax2.legend(loc="upper right",frameon=False)
    ax.grid(alpha=.15);trend.tight_layout()
    folder=Path(tempfile.mkdtemp(prefix="nvidia_fcn_future_"))
    table.to_csv(folder/"all_steps_with_future_flags.csv",index=False)
    future_table.to_csv(folder/"future_city_forecast.csv",index=False)
    fig.savefig(folder/"myanmar_forecast.png",dpi=150)
    trend.savefig(folder/"city_trend.png",dpi=150)
    np.savez_compressed(folder/"regional_forecast.npz",lat=r["lat"],lon=r["lon"],
                        lead_hours=r["lead_hours"],temperature_c=r["temp_c"],wind_kmh=r["wind_kmh"],
                        valid_time_utc=valid.strftime('%Y-%m-%dT%H:%M:%SZ').to_numpy(dtype=str),
                        is_future_at_display=future_mask)
    ahead=int(r["requested_hours_ahead"]) if r.get("requested_hours_ahead") is not None else None
    meta=dict(model="NVIDIA FourCastNet / FCN",forecast_mode=mode,initialization_utc=start.isoformat(),
        total_rollout_hours=hours,requested_hours_ahead=ahead,
        requested_at_utc=r["requested_utc"],requested_target_utc=r["requested_target_utc"],
        display_generated_utc=display_now.isoformat(),final_valid_utc=valid[-1].isoformat(),
        final_valid_myanmar=local[-1].isoformat(),future_row_count=len(future_table),
        city=city,initial_conditions="NOAA GFS via Earth2Studio",gfs_initial_state_url=cycle_urls(start)[0],
        ai_inference=True,synthetic_inputs=False,completed_utc=r["completed_utc"],
        end_to_end_seconds=r["end_to_end_seconds"],gpu=torch.cuda.get_device_name(0),
        grid_lat=float(r["lat"][iy]),grid_lon=float(r["lon"][ix]),
        earth2studio_version=version("earth2studio"),torch_version=torch.__version__,
        locally_validated=False,climate_projection=False)
    (folder/"metadata.json").write_text(json.dumps(meta,indent=2))
    (folder/"storm_alert.json").write_text(json.dumps(storm_alert, indent=2, ensure_ascii=False))
    archive=folder/f"NVIDIA_FCN_Myanmar_{mode.title()}_Forecast.zip"
    with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(folder.iterdir()):
            if p!=archive:z.write(p,arcname=p.name)
    plt.close(fig);plt.close(trend)
    heading="Future NVIDIA AI forecast" if is_future_mode else "NVIDIA AI forecast — initialization-based"
    timing=(f"**Requested future window:** {str(ahead//24)+' day(s) ('+str(ahead)+' h)' if ahead%24==0 else str(ahead)+' h'} · **Model rollout:** {hours} h" if is_future_mode
            else f"**Forecast duration from initialization:** {hours} h")
    interpretation=("The table shows only times later than display generation; the target is rounded forward to the six-hour grid."
                    if is_future_mode else
                    "The table includes the GFS initial state and every forecast step. Historical valid times are retrospective results.")
    eye = storm_alert["coordinates"]
    storm_line = (
        f"**Storm alert:** ACTIVE · {storm_alert['max_sustained_wind_kmh']:.1f} km/h "
        f"at {eye['lat']}°N, {eye['lon']}°E · lead +{storm_alert['lead_hour']}h\n\n"
        if storm_alert["alert_active"]
        else (
            f"**Storm alert:** clear · BoB peak {storm_alert['max_sustained_wind_kmh']:.1f} km/h "
            f"(threshold {CYCLONE_WIND_THRESHOLD_KMH:.0f} km/h; scan Lat 10–22 / Lon 92–96)\n\n"
        )
    )
    status=(f"### {heading} · {city}\n"
        f"**Temperature:** {display_table.temperature_c.iloc[-1]:.1f} °C · **Wind:** {display_table.wind_kmh.iloc[-1]:.1f} km/h\n\n"
        f"{storm_line}"
        f"**Forecast valid:** {local[-1].strftime('%Y-%m-%d %H:%M')} MMT · **Future points:** {len(future_table)}\n\n"
        f"**GFS initialization:** {start.strftime('%Y-%m-%d %H:%M')} UTC · {timing}\n\n"
        f"**Display generated:** {pd.Timestamp(display_now).tz_convert('Asia/Yangon').strftime('%Y-%m-%d %H:%M')} MMT · "
        f"**GPU run completed:** {r['completed_utc']}\n\n{interpretation}")
    return status,fig,trend,display_table,str(archive)



def run_forecast(date_text,hours,city,progress=gr.Progress()):
    progress(.02,desc="Finding published GFS cycle / running future NVIDIA forecast...")
    try:
        start,ahead=validate(date_text,hours,city)
        requested=utc_now()
        rollout_hours,target=future_plan(start,ahead,requested)
        r=dict(predict(start,rollout_hours))
        r.update(forecast_mode="future",requested_hours_ahead=ahead,requested_utc=requested.isoformat(),requested_target_utc=target.isoformat())
        rendered=render_result(start,rollout_hours,city,r)
        progress(1,desc="Future NVIDIA forecast ready")
        return rendered
    except Exception as exc:
        traceback.print_exc()
        return f"### Run failed\n{type(exc).__name__}: {exc}\n\nNo substitute forecast was generated.",None,None,pd.DataFrame(),None


def run_standard_forecast(date_text,hours,city,progress=gr.Progress()):
    progress(.02,desc="Running NVIDIA forecast from the selected initialization...")
    try:
        start,duration=validate_standard(date_text,hours,city)
        requested=utc_now()
        r=dict(predict(start,duration))
        r.update(forecast_mode="standard",requested_hours_ahead=None,
                 requested_utc=requested.isoformat(),
                 requested_target_utc=(start+timedelta(hours=duration)).isoformat())
        output=render_result(start,duration,city,r)
        progress(1,desc="Initialization-based NVIDIA forecast ready")
        return output
    except Exception as exc:
        traceback.print_exc()
        return f"### Run failed\n{type(exc).__name__}: {exc}\n\nNo substitute forecast was generated.",None,None,pd.DataFrame(),None


## Open the showcase dashboard
The dashboard separates the controls, final forecast summary, regional maps, city trends and downloads. Results remain empty until actual NVIDIA GPU inference succeeds. A settings change clears the displayed result; there is no synthetic fallback.

Colab creates a Gradio share link. Anyone who has that link can access the running dashboard. Close the interface after the presentation.

The top-level tabs are **Run NVIDIA AI forecast** and **Predict future weather**. Their controls and results are independent; changing settings in one tab does not clear the other tab. Both share the same serialized GPU inference queue.


In [ ]:
import html


def summary_cards(table):
    if table is None or table.empty:
        return ""
    final=table.iloc[-1]
    future_mode=final.get("forecast_mode","future")=="future"
    timing_card=(("HOURS INTO THE FUTURE",f"{final['hours_ahead_at_display']:.1f}","from display generation") if future_mode else
                 ("MODEL LEAD TIME",str(int(final["lead_hours"])),"hours after initialization"))
    items=[
        ("2 M AIR TEMPERATURE",f"{final['temperature_c']:.1f}","°C"),
        ("10 M WIND SPEED",f"{final['wind_kmh']:.1f}","km/h"),
        timing_card,
        ("VALID IN MYANMAR",str(final['valid_myanmar']).replace(" MMT",""),"UTC +06:30"),
    ]
    return '<div class="forecast-kpis">'+''.join(
        '<div class="forecast-kpi"><div class="kpi-label">'+html.escape(label)+
        '</div><div class="kpi-value">'+html.escape(value)+
        '</div><div class="kpi-unit">'+html.escape(unit)+'</div></div>'
        for label,value,unit in items
    )+'</div>'


def professional_run(date_text,hours,city,progress=gr.Progress()):
    # The original inference function returns only actual GPU-derived results.
    status,maps,trend,table,archive=run_forecast(date_text,hours,city,progress)
    return status,summary_cards(table),maps,trend,table,archive


def professional_standard_run(date_text,hours,city,progress=gr.Progress()):
    status,maps,trend,table,archive=run_standard_forecast(date_text,hours,city,progress)
    return status,summary_cards(table),maps,trend,table,archive


def clear_professional():
    return "Settings updated. Run this tab again to generate matching results.","",None,None,pd.DataFrame(),None


CSS="""
.gradio-container{max-width:1380px!important;font-family:Inter,Arial,sans-serif;}
#showcase-hero{background:linear-gradient(115deg,#102b40,#075b69);border-radius:20px;padding:28px 30px;margin-bottom:12px;}
#showcase-hero h1{color:#fff!important;font-size:32px!important;letter-spacing:-.6px;margin-bottom:8px;}
#showcase-hero p{color:#dcebf0!important;max-width:930px;line-height:1.65;}
#showcase-hero .eyebrow{font-size:12px;letter-spacing:2px;color:#99e7d6!important;}
#source-strip{padding:10px 4px;color:#476173;font-size:13px;margin-bottom:14px;}
.forecast-control-panel{border:1px solid #dce5ec;border-radius:16px;padding:18px;background:#f8fafc;}
.forecast-status{padding:6px 0;}
.forecast-kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;margin:12px 0 18px;}
.forecast-kpi{border:1px solid #d9e6e9;background:#f3f9fa;border-radius:14px;padding:18px;}
.kpi-label{font-size:10px;letter-spacing:1.1px;color:#567080;min-height:28px;}
.kpi-value{font-size:27px;font-weight:700;letter-spacing:-.6px;color:#12394c;line-height:1.3;overflow-wrap:anywhere;}
.forecast-kpi:last-child .kpi-value{font-size:18px;letter-spacing:0;}
.kpi-unit{font-size:12px;color:#55717f;margin-top:6px;}
#showcase-footnote{font-size:12px;color:#64748b;padding:12px 4px;}
@media(max-width:850px){.forecast-kpis{grid-template-columns:repeat(2,minmax(0,1fr));}#showcase-hero{padding:22px;}}
"""
def build_forecast_workspace(mode):
    future=mode=="future"
    action="Predict future weather" if future else "Run NVIDIA AI forecast"
    with gr.Row():
        with gr.Column(scale=1,min_width=280,elem_classes=["forecast-control-panel"]):
            gr.Markdown("### Future weather prediction" if future else "### Initialization-based forecast")
            gr.Markdown("အခုအချိန်ကနေ နောက် ၁–၁၄ ရက်အတွက် ခန့်မှန်းပါ။" if future else "ရွေးထားသော GFS initialization မှ စတင်ခန့်မှန်းပါ။")
            city=gr.Dropdown([name for name in CITIES if name != "Naypyidaw"],value="Yangon",label="City / မြို့")
            date=gr.Textbox(value=DEFAULT_DATE,label="GFS initialization",
                info="AUTO or YYYY-MM-DD HH:MM UTC within the last 30 hours" if future
                else "AUTO or available historical cycle: YYYY-MM-DD HH:MM UTC")
            if future:
                horizon=gr.Dropdown([
                    ("Next 6 hours",6),("Next 12 hours",12),("Next day — 24 hours",24),
                    ("Next 2 days — 48 hours",48),("Next 3 days — 72 hours",72),
                    ("Next 4 days — 96 hours",96),("Next 5 days — 120 hours",120),
                    ("Next 6 days — 144 hours",144),("Next 7 days — 168 hours",168),
                    ("Next 8 days — 192 hours",192),("Next 9 days — 216 hours",216),
                    ("Next 10 days — 240 hours",240),("Next 11 days — 264 hours",264),
                    ("Next 12 days — 288 hours",288),("Next 13 days — 312 hours",312),
                    ("Next 14 days — 336 hours",336),
                ],value=24,label="Future prediction — select next day(s)")
            else:
                horizon=gr.Dropdown([6,12,24,30,48],value=30,label="Forecast hours after initialization (30 h = showcase 0–30 h timeline)")
            button=gr.Button(action,variant="primary",size="lg")
            gr.Markdown(
                "**Future mode:** days are rolling 24-hour periods from request time. The model runs additional steps to bridge the initialization lag. The table shows future points only."
                if future else
                "**Standard mode:** the selected duration is measured from initialization. The table includes the initial state and all forecast steps; historical valid times are retrospective results.")
            gr.Markdown("The first run downloads weights and atmospheric inputs before GPU inference.")
        with gr.Column(scale=3,min_width=500):
            status=gr.Markdown(f"### Ready\nChoose settings and select **{action}**. This tab has its own results.",elem_classes=["forecast-status"])
            cards=gr.HTML()
            with gr.Tabs():
                with gr.Tab("Regional outlook"):
                    maps=gr.Plot(label="Temperature and wind")
                with gr.Tab("City timeline"):
                    trend=gr.Plot(label="City forecast over time")
                with gr.Tab("Forecast data"):
                    table=gr.Dataframe(interactive=False,label="Future forecast points" if future else "Initial state and forecast steps")
                with gr.Tab("Download results"):
                    gr.Markdown("Download forecast arrays, CSV tables, chart images and timestamped provenance for this run.")
                    archive=gr.File(label="Forecast evidence package — ZIP")
    inputs=[date,horizon,city]
    outputs=[status,cards,maps,trend,table,archive]
    callback=professional_run if future else professional_standard_run
    # Both actions share the GPU queue, while controls and outputs are independent.
    button.click(callback,inputs,outputs,concurrency_limit=1,concurrency_id="nvidia_forecasts")
    for control in inputs:
        control.change(clear_professional,None,outputs,queue=False)
    return dict(inputs=inputs,outputs=outputs,button=button,city=city,date=date,horizon=horizon)


with gr.Blocks(theme=gr.themes.Soft(primary_hue="teal",neutral_hue="slate"),css=CSS,
               title="Myanmar Weather Intelligence | NVIDIA GPU Demo") as demo:
    gr.HTML("""<div id="showcase-hero"><p class="eyebrow">NVIDIA EARTH-2 · RESEARCH SHOWCASE</p>
    <h1>Myanmar Weather Intelligence</h1>
    <p>Run initialization-based forecasts or predict future temperature and wind.
    Each workspace has its own settings, maps, city timeline and downloads.</p></div>""")
    gr.Markdown("**MODEL** NVIDIA FourCastNet &nbsp; · &nbsp; **INPUT** NOAA GFS &nbsp; · &nbsp; **EXECUTION** Colab GPU &nbsp; · &nbsp; **TIME ZONE** Myanmar / UTC+06:30",elem_id="source-strip")
    with gr.Tabs():
        with gr.Tab("Run NVIDIA AI forecast",id="standard_forecast"):
            standard_workspace=build_forecast_workspace("standard")
        with gr.Tab("Predict future weather",id="future_forecast"):
            future_workspace=build_forecast_workspace("future")
    with gr.Accordion("Method and interpretation",open=False):
        gr.Markdown("""**How it works:** real GFS atmospheric fields initialize a pretrained NVIDIA FCN model; CUDA inference produces forecast steps every six hours. The dashboard selects regional outputs for Myanmar and surrounding areas.

**Two actions:** Run NVIDIA AI forecast uses the selected hours after initialization and includes the initial state. Predict future weather uses the separate next-day selector: 1–14 rolling 24-hour periods beyond request time, and shows only future points. The endpoint rounds forward to the six-hour grid; forecasts farther ahead are more uncertain. A 14-day run is many GPU steps — use a large GPU and expect a longer wait. Both actions run the same pretrained NVIDIA FCN model on GPU.\n\n**Reading future results:** AUTO checks the newest published GFS initial-state GRIB and index. The future window is measured from request time; the endpoint is rounded forward to the FCN six-hour grid. Only timestamps later than display generation appear in the dashboard table. Exports keep +0 h and earlier model steps with future flags. Those flags describe the saved display-generation time and do not update automatically as wall-clock time passes.

**Scope:** short-range temperature and wind forecasts. Outputs have not been validated against Myanmar observations; this demo does not estimate rainfall or produce a long-term climate projection. Maps include neighboring areas.

**Source:** [NVIDIA Earth2Studio](https://nvidia.github.io/earth2studio/main/) · [FCN implementation](https://github.com/NVIDIA/earth2studio/blob/main/earth2studio/models/px/fcn.py)
""")
    gr.Markdown("Research demonstration · NVIDIA GPU inference · Real atmospheric inputs · Source and run metadata included in exports",elem_id="showcase-footnote")

demo.queue(default_concurrency_limit=1).launch(share=True,debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://cd0f56e7174e380aa2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/tmp/ipykernel_2515/1592245820.py", line 82, in validate_standard
    start=datetime.strptime(text,"%Y-%m-%d %H:%M").replace(tzinfo=timezone.utc)
          ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/_strptime.py", line 789, in _strptime_datetime
    tt, fraction, gmtoff_fraction = _strptime(data_string, format)
                                    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.13/_strptime.py", line 555, in _strptime
    raise ValueError("time data %r does not match format %r" %
                     (data_string, format))
ValueError: time data '2026-10-30' does not match format '%Y-%m-%d %H:%M'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/tmp/ipykernel_2515/1592245820.py", line 244, in run_standard_forecast
    start,duration=validate_standard(date_text,hours,city)
                   ~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^
  

Downloading/loading NVIDIA pretrained FCN weights...


2026-11-02 02:28:01.894 | INFO     | earth2studio.run:deterministic:85 - Running simple workflow!
2026-11-02 02:28:01.895 | INFO     | earth2studio.run:deterministic:92 - Inference device: cuda


Fetching GFS data:   0%|          | 0/26 [00:00<?, ?it/s]

2026-11-02 02:28:03.419 | DEBUG    | earth2studio.data.gfs:fetch_array:395 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20261030/00/atmos/gfs.t00z.pgrb2.0p25.f000 338907157-563037
2026-11-02 02:28:03.422 | DEBUG    | earth2studio.data.gfs:fetch_array:395 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20261030/00/atmos/gfs.t00z.pgrb2.0p25.f000 399753576-830037
2026-11-02 02:28:03.424 | DEBUG    | earth2studio.data.gfs:fetch_array:395 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20261030/00/atmos/gfs.t00z.pgrb2.0p25.f000 203163199-731423
2026-11-02 02:28:03.426 | DEBUG    | earth2studio.data.gfs:fetch_array:395 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20261030/00/atmos/gfs.t00z.pgrb2.0p25.f000 257470964-722858
2026-11-02 02:28:03.429 | DEBUG    | earth2studio.data.gfs:fetch_array:395 - Fetching GFS grib file: noaa-gfs-bdp-pds/gfs.20261030/00/atmos/gfs.t00z.pgrb2.0p25.f000 393384722-944187
2026-11-02 02:28:03.432 | DEBUG    | earth2studio.data.gfs:fetch_array:395 - Fetching GFS 

... (43 outputs left out)

Fetching GFS data: 100%|██████████| 26/26 [00:00<00:00, 39.23it/s]


2026-11-02 03:53:28.293 | SUCCESS  | earth2studio.run:deterministic:154 - Fetched data from GFS
2026-11-02 03:53:28.294 | INFO     | earth2studio.run:deterministic:162 - Inference starting!



Running inference: 100%|██████████| 5/5 [00:00<00:00,  7.08it/s]


2026-11-02 03:53:29.003 | SUCCESS  | earth2studio.run:deterministic:189 - 
Inference complete
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://cd0f56e7174e380aa2.gradio.live


## Professional showcase walkthrough — 3 minutes

1. **Introduce the purpose (30 s):** “Myanmar Weather Intelligence demonstrates regional weather forecasting with a pretrained NVIDIA model running on a GPU.”
2. **Show the inputs (30 s):** identify the model, real GFS atmospheric data, selected city and GFS initialization cycle.
3. **Run and interpret (60 s, excluding downloads/inference):** select AUTO and a 24-hour future window, then show its valid time, temperature and wind values. Explain that the model rolls forward from GFS initialization and the dashboard shows future valid times.
4. **Show evidence (30 s):** open the city timeline and forecast table; download the output package and inspect provenance.
5. **State the evaluation status (30 s):** this is a research demonstration; local forecast accuracy has not yet been measured.

**Presentation preparation:** perform a complete GPU run beforehand, download its evidence package, and retain its timestamps if showing those saved results. A saved run should be described as a previously generated forecast, not as a new live inference.

**Suggested project description:**
> Myanmar Weather Intelligence is a GPU-powered demonstration that applies NVIDIA FourCastNet to real atmospheric data and presents regional temperature and wind forecasts through an interactive dashboard.


## Close demo


In [ ]:
demo.close()


## Troubleshooting and scope
- **Installation failure:** inspect `/content/earth2_fcn_install.log`; do not run later cells with the old package.
- **14-day prediction:** about 56 future six-hour steps, plus extra steps to bridge GFS initialization lag (rollout cap 384 hours). The last valid time rounds forward by less than six hours beyond the requested endpoint. More steps mean more GPU time and memory; longer-range predictions are more uncertain. 14-day forecast accuracy has not been validated here. Download the zip and drop it on the showcase — the UI plays every step.
- **GPU memory:** use a suitable GPU and fresh runtime; a regional output crop does not reduce the global model's state dimensions.
- **Future timestamps:** forecast validity is checked again after inference. Expired results are rejected. Forecast flags in exported files are relative to their recorded display-generation time. A live dashboard is a snapshot, so rerun it to refresh.
- **AUTO cannot find data:** the GFS initial-state GRIB and index may not yet be published, or the network may be unavailable. No future initial conditions are fabricated.
- **Download failure:** model weights and real GFS initial fields must be accessible. Missing upstream data are errors, not an invitation to substitute synthetic inputs.
- **Earlier failed StormScope installation:** this is a separate FCN notebook. Use a fresh runtime rather than carrying partially changed dependencies forward.
- **Weather vs climate:** this notebook does short-range weather rollout. A climate experiment needs a suitable climate model, forcings and evaluation workflow.
- **Preparation validation:** Python syntax, notebook structure, future-cycle selection, rollout planning, future-only filtering, plotting/export behavior and Gradio dashboard construction/callback contracts checked with isolated fixtures. CUDA inference and live browser rendering were not tested in this environment.

Official references:
- https://nvidia.github.io/earth2studio/main/
- https://github.com/NVIDIA/earth2studio/blob/main/earth2studio/models/px/fcn.py
- https://nvidia.github.io/earth2studio/main/modules/models_px/
